# Task 2: Qwen3-ASR-1.7B (CPU version)
Run the 3 cells in order. Runs on CPU, no GPU needed. It is slow, so wait for each cell to finish.

In [ ]:
!pip install -q -U qwen-asr jiwer datasets soundfile psutil

In [ ]:
import torch, time, psutil, os
from qwen_asr import Qwen3ASRModel
t = time.time()
model = Qwen3ASRModel.from_pretrained("Qwen/Qwen3-ASR-1.7B", dtype=torch.float32, device_map="cpu")
print("load time:", round(time.time()-t,1), "s")
print("CPU cores:", os.cpu_count(), "| RAM used GB:", round(psutil.Process().memory_info().rss/1e9,2))

In [ ]:
import re, jiwer, soundfile as sf
from datasets import load_dataset
fix = lambda t: re.sub(r"[^\w\s]", "", t.lower())
for kind in ["clean", "other"]:
    for ex in load_dataset("openslr/librispeech_asr", kind, split="test", streaming=True).take(2):
        sf.write("a.wav", ex["audio"]["array"], 16000)
        sec = len(ex["audio"]["array"]) / 16000
        start = time.time()
        hyp = model.transcribe(audio="a.wav")[0].text
        took = time.time() - start
        print(kind, "| audio", round(sec,1), "s | time", round(took,2), "s | WER", round(jiwer.wer(fix(ex["text"]), fix(hyp)),3), "| CER", round(jiwer.cer(fix(ex["text"]), fix(hyp)),3))
        print("  REF:", ex["text"].lower()); print("  HYP:", hyp)